here are production-grade GitHub Actions you can drop in. They cover CI (lint/tests), CD (build → Artifact Registry → Cloud Run services + Job), Gateway deploy (JWT/OIDC, CORS), a nightly job trigger, and an optional OWASP ZAP API baseline scan.

github workflows CI yaml

In [ ]:
name: CI

on:
  pull_request:
    branches: [ main, dev ]

jobs:
  test-lint-validate:
    runs-on: ubuntu-latest
    timeout-minutes: 25
    strategy:
      matrix:
        python-version: ['3.11']
    steps:
      - name: Checkout
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: ${{ matrix.python-version }}

      - name: Cache pip
        uses: actions/cache@v4
        with:
          path: ~/.cache/pip
          key: pip-${{ runner.os }}-${{ hashFiles('requirements.txt') }}

      - name: Install deps (prod + test)
        run: |
          python -m pip install --upgrade pip
          pip install -r requirements.txt
          pip install pytest ruff authlib python-jose cachetools
          # OpenAPI linter
          npm i -g @stoplight/spectral

      - name: Ruff Lint
        run: ruff check .

      - name: Unit tests
        env:
          # Fake keys; unit tests should not call real APIs
          OPENAI_API_KEY: "test"
          LANGCHAIN_API_KEY: "test"
          # Optional OIDC mock values for security tests
          OIDC_ISSUER: "https://idp.example.com"
          OIDC_AUDIENCE: "client-id-test"
          OIDC_JWKS_URI: "http://localhost/.well-known/jwks.json"
        run: pytest -q

      - name: Lint OpenAPI (API Gateway spec)
        if: hashFiles('gateway/openapi.yaml') != ''
        run: spectral lint gateway/openapi.yaml


github/workflows/cd.yml — CD to GCP on main (Cloud Run + Job)

In [ ]:
name: CD

on:
  push:
    branches: [ main ]

env:
  REGION: us-central1
  REPO: ai-images
  IMAGE_NAME: security-pipeline
  SERVICE_ORCH: orchestrator-api
  SERVICE_CHAT: chatbot-api
  JOB_NAME: orchestrator-job

permissions:
  contents: read
  id-token: write

jobs:
  build-deploy:
    runs-on: ubuntu-latest
    timeout-minutes: 45

    steps:
      - name: Checkout
        uses: actions/checkout@v4

      # Auth to GCP via Workload Identity Federation (recommended)
      - name: Auth to GCP
        uses: google-github-actions/auth@v2
        with:
          workload_identity_provider: ${{ secrets.GCP_WORKLOAD_IDP }}
          service_account: ${{ secrets.GCP_SERVICE_ACCOUNT_EMAIL }}

      - name: Setup gcloud
        uses: google-github-actions/setup-gcloud@v2
        with:
          project_id: ${{ secrets.GCP_PROJECT_ID }}
          install_components: 'beta'

      - name: Configure Docker for Artifact Registry
        run: gcloud auth configure-docker ${{ env.REGION }}-docker.pkg.dev

      - name: Build & Push image
        run: |
          IMAGE="${{ env.REGION }}-docker.pkg.dev/${{ secrets.GCP_PROJECT_ID }}/${{ env.REPO }}/${{ env.IMAGE_NAME }}:${GITHUB_SHA::7}"
          echo "IMAGE=$IMAGE" >> $GITHUB_ENV
          docker build -t "$IMAGE" .
          docker push "$IMAGE"

      - name: Deploy Cloud Run - orchestrator-api
        run: |
          gcloud run deploy ${{ env.SERVICE_ORCH }} \
            --image "${IMAGE}" \
            --region "${{ env.REGION }}" \
            --allow-unauthenticated \
            --set-env-vars "PORT=8080,LANGCHAIN_TRACING_V2=true,RAW_BUCKET=${{ secrets.RAW_BUCKET }},UNIFIED_BUCKET=${{ secrets.UNIFIED_BUCKET }}" \
            --set-env-vars "OIDC_ISSUER=${{ secrets.OIDC_ISSUER }},OIDC_AUDIENCE=${{ secrets.OIDC_AUDIENCE }},OIDC_JWKS_URI=${{ secrets.OIDC_JWKS_URI }},ALLOWED_ORIGINS=${{ secrets.ALLOWED_ORIGINS }}" \
            --set-secrets "OPENAI_API_KEY=OPENAI_API_KEY:latest,LANGCHAIN_API_KEY=LANGCHAIN_API_KEY:latest"

      - name: Deploy Cloud Run - chatbot-api
        run: |
          gcloud run deploy ${{ env.SERVICE_CHAT }} \
            --image "${IMAGE}" \
            --region "${{ env.REGION }}" \
            --allow-unauthenticated \
            --command uvicorn --args api.main_chatbot:app,--host,0.0.0.0,--port,8080 \
            --set-env-vars "PORT=8080,LANGCHAIN_TRACING_V2=true,UNIFIED_BUCKET=${{ secrets.UNIFIED_BUCKET }}" \
            --set-env-vars "OIDC_ISSUER=${{ secrets.OIDC_ISSUER }},OIDC_AUDIENCE=${{ secrets.OIDC_AUDIENCE }},OIDC_JWKS_URI=${{ secrets.OIDC_JWKS_URI }},ALLOWED_ORIGINS=${{ secrets.ALLOWED_ORIGINS }}" \
            --set-secrets "OPENAI_API_KEY=OPENAI_API_KEY:latest,LANGCHAIN_API_KEY=LANGCHAIN_API_KEY:latest"

      - name: Ensure Cloud Run Job exists (create or update)
        run: |
          set -e
          IMAGE="${IMAGE}"
          REGION="${{ env.REGION }}"
          JOB="${{ env.JOB_NAME }}"

          if gcloud run jobs describe "$JOB" --region "$REGION" >/dev/null 2>&1; then
            gcloud run jobs update "$JOB" \
              --image "$IMAGE" \
              --region "$REGION" \
              --command python --args workflow_orchestrator_main.py \
              --set-env-vars "LANGCHAIN_TRACING_V2=true,RAW_BUCKET=${{ secrets.RAW_BUCKET }},UNIFIED_BUCKET=${{ secrets.UNIFIED_BUCKET }}" \
              --set-env-vars "OIDC_ISSUER=${{ secrets.OIDC_ISSUER }},OIDC_AUDIENCE=${{ secrets.OIDC_AUDIENCE }},OIDC_JWKS_URI=${{ secrets.OIDC_JWKS_URI }}" \
              --set-secrets "OPENAI_API_KEY=OPENAI_API_KEY:latest,LANGCHAIN_API_KEY=LANGCHAIN_API_KEY:latest"
          else
            gcloud run jobs create "$JOB" \
              --image "$IMAGE" \
              --region "$REGION" \
              --command python --args workflow_orchestrator_main.py \
              --set-env-vars "LANGCHAIN_TRACING_V2=true,RAW_BUCKET=${{ secrets.RAW_BUCKET }},UNIFIED_BUCKET=${{ secrets.UNIFIED_BUCKET }}" \
              --set-env-vars "OIDC_ISSUER=${{ secrets.OIDC_ISSUER }},OIDC_AUDIENCE=${{ secrets.OIDC_AUDIENCE }},OIDC_JWKS_URI=${{ secrets.OIDC_JWKS_URI }}" \
              --set-secrets "OPENAI_API_KEY=OPENAI_API_KEY:latest,LANGCHAIN_API_KEY=LANGCHAIN_API_KEY:latest"
          fi

      - name: Smoke test /healthz
        run: |
          ORCH_URL=$(gcloud run services describe ${{ env.SERVICE_ORCH }} --region ${{ env.REGION }} --format='value(status.url)')
          curl -fsS "${ORCH_URL}/healthz"


.github/workflows/gateway-deploy.yml — Deploy API Gateway (JWT/OIDC, CORS)

This workflow updates the Gateway whenever your Cloud Run URL or OpenAPI spec changes. It patches x-google-backend.address to your latest service URL, versions the config by commit SHA, and switches the Gateway to the new config.

In [ ]:
name: Deploy API Gateway

on:
  push:
    branches: [ main ]
    paths:
      - 'gateway/openapi.yaml'
  workflow_dispatch: {}

env:
  REGION: us-central1
  API_ID: chatbot-api
  GATEWAY_ID: chatbot-gw

permissions:
  contents: read
  id-token: write

jobs:
  deploy-gateway:
    runs-on: ubuntu-latest
    timeout-minutes: 20

    steps:
      - name: Checkout
        uses: actions/checkout@v4

      - name: Auth to GCP
        uses: google-github-actions/auth@v2
        with:
          workload_identity_provider: ${{ secrets.GCP_WORKLOAD_IDP }}
          service_account: ${{ secrets.GCP_SERVICE_ACCOUNT_EMAIL }}

      - name: Setup gcloud
        uses: google-github-actions/setup-gcloud@v2
        with:
          project_id: ${{ secrets.GCP_PROJECT_ID }}

      - name: Get Cloud Run URL
        id: runurl
        run: |
          URL=$(gcloud run services describe chatbot-api --region ${{ env.REGION }} --format='value(status.url)')
          echo "BACKEND_URL=$URL" >> $GITHUB_ENV

      - name: Prepare OpenAPI (patch backend address + OIDC)
        run: |
          cp gateway/openapi.yaml gateway/openapi.patched.yaml
          # Replace placeholder <cloud-run-url> with the actual Cloud Run URL
          sed -i "s#https://<cloud-run-url>#${BACKEND_URL}#g" gateway/openapi.patched.yaml
          # Optional: patch issuer/audience/JWKS if you keep placeholders in YAML
          sed -i "s#YOUR_IDP_ISSUER#${{ secrets.OIDC_ISSUER }}#g" gateway/openapi.patched.yaml
          sed -i "s#YOUR_OIDC_CLIENT_ID#${{ secrets.OIDC_AUDIENCE }}#g" gateway/openapi.patched.yaml
          sed -i "s#https://YOUR_IDP/.well-known/jwks.json#${{ secrets.OIDC_JWKS_URI }}#g" gateway/openapi.patched.yaml

      - name: Create API (idempotent)
        run: gcloud api-gateway apis create ${{ env.API_ID }} --project=${{ secrets.GCP_PROJECT_ID }} || true

      - name: Create API Config (versioned by SHA)
        run: |
          CFG="cfg-${GITHUB_SHA::7}"
          gcloud api-gateway api-configs create "$CFG" \
            --api=${{ env.API_ID }} \
            --openapi-spec=gateway/openapi.patched.yaml \
            --project=${{ secrets.GCP_PROJECT_ID }} \
            --backend-auth-service-account=${{ secrets.GATEWAY_BACKEND_SA }}
          echo "CFG=$CFG" >> $GITHUB_ENV

      - name: Create Gateway (first time) or Update to new config
        run: |
          if gcloud api-gateway gateways describe ${{ env.GATEWAY_ID }} --location=${{ env.REGION }} >/dev/null 2>&1; then
            gcloud api-gateway gateways update ${{ env.GATEWAY_ID }} \
              --api=${{ env.API_ID }} --api-config=${CFG} \
              --location=${{ env.REGION }}
          else
            gcloud api-gateway gateways create ${{ env.GATEWAY_ID }} \
              --api=${{ env.API_ID }} --api-config=${CFG} \
              --location=${{ env.REGION }}
          fi

      - name: Output Gateway URL
        run: |
          GW_URL=$(gcloud api-gateway gateways describe ${{ env.GATEWAY_ID }} --location=${{ env.REGION }} --format='value(defaultHostname)')
          echo "Gateway URL: https://${GW_URL}"


.github/workflows/nightly-job.yml — Nightly Cloud Run Job

In [ ]:
name: Nightly Orchestrator Job

on:
  schedule:
    - cron: '0 6 * * *'  # 2 AM ET
  workflow_dispatch: {}

permissions:
  id-token: write
  contents: read

jobs:
  run-orchestrator:
    runs-on: ubuntu-latest
    timeout-minutes: 20
    steps:
      - name: Auth to GCP
        uses: google-github-actions/auth@v2
        with:
          workload_identity_provider: ${{ secrets.GCP_WORKLOAD_IDP }}
          service_account: ${{ secrets.GCP_SERVICE_ACCOUNT_EMAIL }}

      - name: Setup gcloud
        uses: google-github-actions/setup-gcloud@v2
        with:
          project_id: ${{ secrets.GCP_PROJECT_ID }}

      - name: Execute Cloud Run Job
        run: gcloud run jobs execute orchestrator-job --region us-central1


.github/workflows/zap-baseline.yml — (Optional) OWASP ZAP baseline on Gateway

In [ ]:
name: ZAP Baseline Scan

on:
  workflow_dispatch: {}
  schedule:
    - cron: '30 6 * * 1'  # weekly

jobs:
  zap-scan:
    runs-on: ubuntu-latest
    timeout-minutes: 45
    steps:
      - name: Auth to GCP
        uses: google-github-actions/auth@v2
        with:
          workload_identity_provider: ${{ secrets.GCP_WORKLOAD_IDP }}
          service_account: ${{ secrets.GCP_SERVICE_ACCOUNT_EMAIL }}

      - name: Setup gcloud
        uses: google-github-actions/setup-gcloud@v2
        with:
          project_id: ${{ secrets.GCP_PROJECT_ID }}

      - name: Get Gateway URL
        id: gw
        run: |
          GW_URL=$(gcloud api-gateway gateways describe chatbot-gw --location=us-central1 --format='value(defaultHostname)')
          echo "GW=https://${GW_URL}" >> $GITHUB_ENV

      - name: ZAP Baseline
        uses: zaproxy/action-baseline@v0.13.0
        with:
          target: ${{ env.GW }}
          rules_file_name: ''
          cmd_options: '-a -m 5'  # aggressive, 5 min max
